# Fix Updated Data, To current time.

This notebook verifies that merged cryptocurrency data matches training data format and creates missing technical indicators using helper programs from the notebooks/helpers folder.

**Objectives:**
- Load merged data from `data/interim/merged/`
- Compare with training data format from `data/processed/`
- Identify missing indicators
- Calculate missing indicators using helper functions
- Validate calculations
- Save enhanced data

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add helpers folder to path
sys.path.append('./helpers')

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully!
Pandas version: 2.3.3
NumPy version: 2.3.5


## Load Helper Programs from Notebook Folder

Import indicator calculation functions from the helpers module

In [2]:
# Import helper functions for indicator calculations
try:
    from indicators import (
        calculate_sma_ema, 
        plot_sma_ema, 
        calculate_rsi, 
        calculate_macd
    )
    from indicators_helper import add_technical_indicators, validate_indicators
    print("✓ Successfully imported all helper functions")
except ImportError as e:
    print(f"Note: Some indicators not available: {e}")
    print("Will implement indicator calculations inline...")

# Define data directories
merged_data_dir = '../data/interim/merged'
processed_data_dir = '../data/processed'

# Create output directory if it doesn't exist
os.makedirs(processed_data_dir, exist_ok=True)
print(f"\nData directories:")
print(f"  Merged: {merged_data_dir}")
print(f"  Processing: {processed_data_dir}")

✓ Successfully imported all helper functions

Data directories:
  Merged: ../data/interim/merged
  Processing: ../data/processed


## Load Merged Data & Compare with Training Data

Load the merged cryptocurrency data and identify which indicators are missing.

In [3]:
# Load merged data using helper function
from helpers import load_data_csv_files

print("Loading merged data from helpers function...\n")
raw_merged_data = load_data_csv_files(merged_data_dir)

# Clean up naming: remove '_merged' suffix and convert Date to datetime
merged_data = {}
for raw_name, df in raw_merged_data.items():
    crypto_name = raw_name.replace('_merged', '')
    df['Date'] = pd.to_datetime(df['Date'])
    merged_data[crypto_name] = df
    print(f"  ✓ {crypto_name:20s} - {len(df):5d} rows")

print(f"\nTotal cryptocurrencies loaded: {len(merged_data)}")

Loading merged data from helpers function...

Loaded: avalanche_merged.csv
Loaded: axie_infinity_merged.csv
Loaded: binance_coin_merged.csv
Loaded: bitcoin_merged.csv
Loaded: chainlink_merged.csv
Loaded: ethereum_merged.csv
Loaded: injective_merged.csv
Loaded: litecoin_merged.csv
Loaded: maker_merged.csv
Loaded: render_merged.csv
Loaded: solana_merged.csv
Loaded: the_graph_merged.csv
Loaded: toncoin_merged.csv
Loaded: tron_merged.csv
  ✓ avalanche            -  2050 rows
  ✓ axie_infinity        -  2005 rows
  ✓ binance_coin         -  3096 rows
  ✓ bitcoin              -  4245 rows
  ✓ chainlink            -  3096 rows
  ✓ ethereum             -  3096 rows
  ✓ injective            -  2019 rows
  ✓ litecoin             -  4245 rows
  ✓ maker                -  3085 rows
  ✓ render               -  2136 rows
  ✓ solana               -  2213 rows
  ✓ the_graph            -  1373 rows
  ✓ toncoin              -  1709 rows
  ✓ tron                 -  3096 rows

Total cryptocurrencies loaded

In [4]:
# Load one example training file to see what columns we need
training_example = pd.read_csv(os.path.join(processed_data_dir, 'bitcoin_processed.csv'))
print("Training Data Columns (bitcoin_processed.csv):")
print(f"  {list(training_example.columns)}\n")

# Check merged data columns
print("Merged Data Columns (bitcoin_merged.csv):")
merged_example = merged_data.get('bitcoin')
print(f"  {list(merged_example.columns)}\n")

# Identify missing indicators
merged_cols = set(merged_example.columns)
training_cols = set(training_example.columns)
missing_indicators = training_cols - merged_cols

print("Missing Indicators:")
for indicator in sorted(missing_indicators):
    print(f"  - {indicator}")

Training Data Columns (bitcoin_processed.csv):
  ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA_7', 'SMA_20', 'SMA_50', 'EMA_7', 'EMA_20', 'EMA_50', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'Daily_Return', 'Weekly_Return', 'Monthly_Return', 'Volatility_30']

Merged Data Columns (bitcoin_merged.csv):
  ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']

Missing Indicators:
  - Daily_Return
  - EMA_20
  - EMA_50
  - EMA_7
  - MACD
  - MACD_Histogram
  - MACD_Signal
  - Monthly_Return
  - RSI_14
  - SMA_20
  - SMA_50
  - SMA_7
  - Volatility_30
  - Weekly_Return


## Create Missing Technical Indicators

Calculate all missing indicators using helper functions and mathematical formulas.

In [5]:
# Apply technical indicators using helper function
print("Calculating technical indicators for all cryptocurrencies...\n")
processed_data = {}
for crypto_name in sorted(merged_data.keys()):
    df = merged_data[crypto_name].copy()
    df_with_indicators = add_technical_indicators(df)
    processed_data[crypto_name] = df_with_indicators
    print(f"  ✓ {crypto_name:20s} - Added 14 indicators")

print(f"\n✓ Technical indicators calculated for {len(processed_data)} cryptocurrencies")

Calculating technical indicators for all cryptocurrencies...

  ✓ avalanche            - Added 14 indicators
  ✓ axie_infinity        - Added 14 indicators
  ✓ binance_coin         - Added 14 indicators
  ✓ bitcoin              - Added 14 indicators
  ✓ chainlink            - Added 14 indicators
  ✓ ethereum             - Added 14 indicators
  ✓ injective            - Added 14 indicators
  ✓ litecoin             - Added 14 indicators
  ✓ maker                - Added 14 indicators
  ✓ render               - Added 14 indicators
  ✓ solana               - Added 14 indicators
  ✓ the_graph            - Added 14 indicators
  ✓ toncoin              - Added 14 indicators
  ✓ tron                 - Added 14 indicators

✓ Technical indicators calculated for 14 cryptocurrencies


## Validate Indicators

Verify that indicators were calculated correctly with no NaN values where unexpected.

In [6]:
# Run validation on all cryptocurrencies using helper function
print("Validating technical indicators...\n")
validation_results = {}
total_issues = 0

for crypto_name in sorted(processed_data.keys()):
    df = processed_data[crypto_name]
    issues = validate_indicators(df, crypto_name)
    validation_results[crypto_name] = issues
    
    if issues:
        total_issues += len(issues)
        print(f"  ⚠ {crypto_name:20s} - {len(issues)} issues found:")
        for issue in issues:
            print(f"      - {issue}")
    else:
        print(f"  ✓ {crypto_name:20s} - All validations passed")

print(f"\n✓ Validation complete: {total_issues} total issues found")

Validating technical indicators...

  ✓ avalanche            - All validations passed
  ✓ axie_infinity        - All validations passed
  ✓ binance_coin         - All validations passed
  ✓ bitcoin              - All validations passed
  ✓ chainlink            - All validations passed
  ✓ ethereum             - All validations passed
  ✓ injective            - All validations passed
  ✓ litecoin             - All validations passed
  ✓ maker                - All validations passed
  ✓ render               - All validations passed
  ✓ solana               - All validations passed
  ✓ the_graph            - All validations passed
  ✓ toncoin              - All validations passed
  ✓ tron                 - All validations passed

✓ Validation complete: 0 total issues found


## Save Processed Data

Combine indicators with merged data and save to processed folder in training format.

In [7]:
# Define the expected column order (matching training data format)
expected_columns = [
    'Date', 'Close', 'High', 'Low', 'Open', 'Volume',
    'SMA_7', 'SMA_20', 'SMA_50', 'EMA_7', 'EMA_20', 'EMA_50',
    'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Histogram',
    'Daily_Return', 'Weekly_Return', 'Monthly_Return', 'Volatility_30'
]

# Save enhanced data
print("Saving enhanced data to data/processed/...\n")
saved_count = 0
failed_count = 0

for crypto_name in sorted(processed_data.keys()):
    try:
        df = processed_data[crypto_name]
        
        # Reorder columns to match expected format
        cols_to_save = [col for col in expected_columns if col in df.columns]
        df_save = df[cols_to_save].copy()
        
        # Save to processed folder
        output_path = os.path.join(processed_data_dir, f'{crypto_name}_processed.csv')
        df_save.to_csv(output_path, index=False)
        saved_count += 1
        print(f"  ✓ {crypto_name:20s} - Saved ({len(df_save)} rows, {len(cols_to_save)} columns)")
        
    except Exception as e:
        failed_count += 1
        print(f"  ✗ {crypto_name:20s} - Error: {str(e)}")

print(f"\n{'='*60}")
print(f"✓ Saved: {saved_count}/{len(processed_data)} cryptocurrencies")
if failed_count > 0:
    print(f"✗ Failed: {failed_count}")
print(f"{'='*60}")
print(f"\nProcessed files saved to: {processed_data_dir}")
print("Files are now in training data format with all technical indicators!")

Saving enhanced data to data/processed/...

  ✓ avalanche            - Saved (2050 rows, 20 columns)
  ✓ axie_infinity        - Saved (2005 rows, 20 columns)
  ✓ binance_coin         - Saved (3096 rows, 20 columns)
  ✓ bitcoin              - Saved (4245 rows, 20 columns)
  ✓ chainlink            - Saved (3096 rows, 20 columns)
  ✓ ethereum             - Saved (3096 rows, 20 columns)
  ✓ injective            - Saved (2019 rows, 20 columns)
  ✓ litecoin             - Saved (4245 rows, 20 columns)
  ✓ maker                - Saved (3085 rows, 20 columns)
  ✓ render               - Saved (2136 rows, 20 columns)
  ✓ solana               - Saved (2213 rows, 20 columns)
  ✓ the_graph            - Saved (1373 rows, 20 columns)
  ✓ toncoin              - Saved (1709 rows, 20 columns)
  ✓ tron                 - Saved (3096 rows, 20 columns)

✓ Saved: 14/14 cryptocurrencies

Processed files saved to: ../data/processed
Files are now in training data format with all technical indicators!
